# 🎤 CYPHER AI — Google Colab Free T4 GPU Backend

This notebook runs the **CYPHER AI backend** on a free Google Colab T4 GPU.
It loads a quantized **Llama-3.1-8B-Instruct** model, exposes an OpenAI-compatible FastAPI server, and creates a public secure HTTPS tunnel via **Cloudflare Tunnel** (no account or token required).

### Step 1: Check GPU Acceleration
Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [ ]:
!nvidia-smi

### Step 2: Install Dependencies
Install FastAPI, Uvicorn, Transformers, Accelerate, BitsAndBytes, and Cloudflared for tunneling.

In [ ]:
!pip install -q fastapi uvicorn pydantic transformers accelerate bitsandbytes pycloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

### Step 3: Start the FastAPI Server with Model & Public Cloudflare Tunnel

In [ ]:
import nest_asyncio
import subprocess
import time
import re
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

app = FastAPI(title="Cypher Colab LLM Backend")

MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
print("Loading model on T4 GPU (4-bit quantized)... This takes ~2 mins.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.float16
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

class ChatReq(BaseModel):
    messages: list[dict]
    temperature: float = 0.7

@app.post("/chat")
def chat_endpoint(req: ChatReq):
    prompt = tokenizer.apply_chat_template(req.messages, tokenize=False, add_generation_prompt=True)
    outputs = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=req.temperature)
    generated_text = outputs[0]["generated_text"][len(prompt):].strip()
    return {"reply": generated_text}

@app.get("/health")
def health():
    return {"status": "ok", "gpu": torch.cuda.get_device_name(0)}

# Start cloudflared tunnel in background
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
time.sleep(4)

print("\n🔥 CYPHER COLAB TUNNEL READY! 🔥")
print("Copy your Public URL and paste it into the CYPHER Web App Settings:")
print("----------------------------------------------------------------")

# Read tunnel URL from stderr
for line in tunnel_proc.stderr:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        print(f"===> PUBLIC URL: {match.group(0)}")
        break

import uvicorn
uvicorn.run(app, host="127.0.0.1", port=8000)